In [1]:
import os
from pathlib import Path
import pandas as pd

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
RAW = DATA_DIR / "raw" / "part_b"
OUT = DATA_DIR / "processed" / "part_b_1319"
SBP = OUT / "sbp"; SBP.mkdir(parents=True, exist_ok=True)
AE = OUT / "enriched"; AE.mkdir(parents=True, exist_ok=True)
YEARS = list(range(2013, 2020))
KEYS = ["Rndrng_NPI", "year", "Rndrng_Prvdr_Type", "Place_Of_Srvc"]

def raw_path(kind, year):
    name = f"Medicare_Physician_Other_Practitioners_by_{kind}_{year}"
    return RAW / name / f"{name}.csv"

In [2]:
def sbp_features(cols):
    return [c for c in cols
            if c.startswith(("Tot_", "Drug_", "Med_", "Bene_"))
            and not c.startswith("Bene_Race_") and not c.endswith("Sprsn_Ind")]

hdr = {y: list(pd.read_csv(raw_path("Provider", y), nrows=0, encoding="latin-1").columns) for y in YEARS}
feats = {y: sbp_features(hdr[y]) for y in YEARS}
for y in YEARS:
    print(y, len(feats[y]))
print("\nKept (2019):", feats[2019])
print("\nDropped (2019):", [c for c in hdr[2019] if c not in feats[2019]])
COMMON = [c for c in feats[2019] if all(c in feats[y] for y in YEARS)]
print("\nFeatures present in all years:", len(COMMON))

2013 31
2014 31
2015 31
2016 31
2017 56
2018 56
2019 56

Kept (2019): ['Tot_HCPCS_Cds', 'Tot_Benes', 'Tot_Srvcs', 'Tot_Sbmtd_Chrg', 'Tot_Mdcr_Alowd_Amt', 'Tot_Mdcr_Pymt_Amt', 'Tot_Mdcr_Stdzd_Amt', 'Drug_Tot_HCPCS_Cds', 'Drug_Tot_Benes', 'Drug_Tot_Srvcs', 'Drug_Sbmtd_Chrg', 'Drug_Mdcr_Alowd_Amt', 'Drug_Mdcr_Pymt_Amt', 'Drug_Mdcr_Stdzd_Amt', 'Med_Tot_HCPCS_Cds', 'Med_Tot_Benes', 'Med_Tot_Srvcs', 'Med_Sbmtd_Chrg', 'Med_Mdcr_Alowd_Amt', 'Med_Mdcr_Pymt_Amt', 'Med_Mdcr_Stdzd_Amt', 'Bene_Avg_Age', 'Bene_Age_LT_65_Cnt', 'Bene_Age_65_74_Cnt', 'Bene_Age_75_84_Cnt', 'Bene_Age_GT_84_Cnt', 'Bene_Feml_Cnt', 'Bene_Male_Cnt', 'Bene_Dual_Cnt', 'Bene_Ndual_Cnt', 'Bene_CC_BH_ADHD_OthCD_V1_Pct', 'Bene_CC_BH_Alcohol_Drug_V1_Pct', 'Bene_CC_BH_Tobacco_V1_Pct', 'Bene_CC_BH_Alz_NonAlzdem_V2_Pct', 'Bene_CC_BH_Anxiety_V1_Pct', 'Bene_CC_BH_Bipolar_V1_Pct', 'Bene_CC_BH_Mood_V2_Pct', 'Bene_CC_BH_Depress_V1_Pct', 'Bene_CC_BH_PD_V1_Pct', 'Bene_CC_BH_PTSD_V1_Pct', 'Bene_CC_BH_Schizo_OthPsy_V1_Pct', 'Bene_CC_PH_Asthma_

In [3]:
raw_sbp, miss, nonnum = {}, {}, {}
for y in YEARS:
    df = pd.read_csv(raw_path("Provider", y), usecols=["Rndrng_NPI"] + feats[y],
                     dtype={"Rndrng_NPI": str}, encoding="latin-1")
    raw_sbp[y] = df
    miss[y] = df[feats[y]].isna().mean().round(3)
    nonnum[y] = [c for c in feats[y] if not pd.api.types.is_numeric_dtype(df[c])]
miss = pd.DataFrame(miss)
print("Non-numeric columns:", {y: v for y, v in nonnum.items() if v} or "none")
print("\nColumns >50% missing in any year:\n", miss[(miss > 0.5).any(axis=1)])
miss.to_csv(OUT / "audit" / "sbp_missing_by_year.csv")

Non-numeric columns: none

Columns >50% missing in any year:
                                  2013  2014  2015  2016   2017   2018   2019
Bene_CC_BH_ADHD_OthCD_V1_Pct      NaN   NaN   NaN   NaN  0.635  0.636  0.645
Bene_CC_BH_Bipolar_V1_Pct         NaN   NaN   NaN   NaN  0.508  0.510  0.514
Bene_CC_BH_PD_V1_Pct              NaN   NaN   NaN   NaN  0.619  0.628  0.636
Bene_CC_BH_PTSD_V1_Pct            NaN   NaN   NaN   NaN  0.636  0.641  0.645
Bene_CC_BH_Schizo_OthPsy_V1_Pct   NaN   NaN   NaN   NaN  0.540  0.547  0.555
Bene_CC_PH_Parkinson_V2_Pct       NaN   NaN   NaN   NaN  0.584  0.588  0.593


In [4]:
for y in YEARS:
    df = raw_sbp[y][raw_sbp[y]["Rndrng_NPI"].notna()].copy()
    bad = 0
    for c in nonnum[y]:
        before = df[c].isna().sum()
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")
        bad += df[c].isna().sum() - before
    df[feats[y]] = df[feats[y]].fillna(0)
    df["year"] = y
    assert not df["Rndrng_NPI"].duplicated().any()
    df.to_parquet(SBP / f"part_b_sbp_{y}.parquet", index=False)
    print(y, df.shape, "| values that failed to convert:", bad)
del raw_sbp

2013 (956251, 33) | values that failed to convert: 0
2014 (986657, 33) | values that failed to convert: 0
2015 (1019377, 33) | values that failed to convert: 0
2016 (1053958, 33) | values that failed to convert: 0
2017 (1088687, 58) | values that failed to convert: 0
2018 (1121462, 58) | values that failed to convert: 0
2019 (1155870, 58) | values that failed to convert: 0


In [5]:
agg_all = pd.read_parquet(OUT / "part_b_agg_pos_2013_2019.parquet")
AGG_FEATS = [c for c in agg_all.columns if c not in KEYS + ["Rndrng_Prvdr_Ent_Cd"]]
parts = []
for y in YEARS:
    a = agg_all[agg_all["year"] == y]
    s = pd.read_parquet(SBP / f"part_b_sbp_{y}.parquet")
    m = a.merge(s, on=["Rndrng_NPI", "year"], how="inner", validate="many_to_one")
    print(y, "| agg rows:", len(a), "| joined:", len(m), "| lost:", len(a) - len(m),
          "| by-Provider NPIs with no claims rows:", int((~s["Rndrng_NPI"].isin(a["Rndrng_NPI"])).sum()))
    m.to_parquet(AE / f"part_b_ae_{y}.parquet", index=False)
    parts.append(m[KEYS + ["Rndrng_Prvdr_Ent_Cd"] + AGG_FEATS + COMMON])

2013 | agg rows: 1142101 | joined: 1142101 | lost: 0 | by-Provider NPIs with no claims rows: 46691
2014 | agg rows: 1167606 | joined: 1167606 | lost: 0 | by-Provider NPIs with no claims rows: 48516
2015 | agg rows: 1198535 | joined: 1198535 | lost: 0 | by-Provider NPIs with no claims rows: 51044
2016 | agg rows: 1244353 | joined: 1244353 | lost: 0 | by-Provider NPIs with no claims rows: 53034
2017 | agg rows: 1275976 | joined: 1275976 | lost: 0 | by-Provider NPIs with no claims rows: 55776
2018 | agg rows: 1304939 | joined: 1304939 | lost: 0 | by-Provider NPIs with no claims rows: 60184
2019 | agg rows: 1338209 | joined: 1338209 | lost: 0 | by-Provider NPIs with no claims rows: 62503


In [6]:
ae = pd.concat(parts, ignore_index=True)
print(ae.shape, "| NaN:", int(ae.isna().sum().sum()))
print("Total:", len(ae), "| paper AE: 8,669,497")
ae.to_parquet(OUT / "part_b_ae_common_2013_2019.parquet", index=False)

(8671719, 66) | NaN: 0
Total: 8671719 | paper AE: 8,669,497
